# Bibliotecas

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

import ipywidgets as widgets # código feito por IA
from IPython.display import display, clear_output # código feito por IA

pd.set_option('display.max_rows', None)

# Carregamento do JSON

In [2]:
with open("data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
print(df.shape)  # deve dar (6674, 7)

(6674, 7)


# Visualização dos dados

In [3]:
#Código feito com IA para auxiliar na visualização dos dados e identificação de problemas do JSON, para posteriormente podermos corrigir com código.
bairros_ordenados = sorted(df["bairro"].unique())

# Mapeamento com índice numérico (1 a 94) para facilitar a seleção
opcoes_bairros = [(f"{i+1} - {bairro}", bairro) for i, bairro in enumerate(bairros_ordenados)]

# 3. Criação dos Widgets Interativos
dropdown_bairro = widgets.Dropdown(
    options=opcoes_bairros,
    value=bairros_ordenados[17],
    description="Bairro:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

slider_niveis = widgets.IntSlider(
    value=71,
    min=1,
    max=df["nivel_m"].nunique(),
    step=1,
    description="Qtd. Níveis:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

output_area = widgets.Output()

# 4. Função de atualização dinâmica dos resultados
def atualizar_exibicao(change=None):
    bairro_selecionado = dropdown_bairro.value
    qtd_niveis = slider_niveis.value
    
    # Filtrar o DataFrame pelo bairro selecionado e ordenar pelo nível do rio (m)
    df_bairro = df[df["bairro"] == bairro_selecionado].sort_values(by="nivel_m")
    resultado = df_bairro.head(qtd_niveis)
    
    with output_area:
        clear_output(wait=True)
        print(f"Bairro selecionado: {bairro_selecionado}")
        print(f"Exibindo os primeiros {len(resultado)} níveis de inundação:")
        display(resultado)

# Vincular os eventos dos widgets à função de atualização
dropdown_bairro.observe(atualizar_exibicao, names='value')
slider_niveis.observe(atualizar_exibicao, names='value')

# 5. Renderizar a interface interativa no Notebook
interface = widgets.VBox([
    widgets.HTML(value="### Painel Interativo de Análise de Inundação"),
    dropdown_bairro,
    slider_niveis,
    widgets.HTML(value="<hr/>"),
    output_area
])

display(interface)
atualizar_exibicao()

# IDENTIFICAÇÃO DE ERROS

In [4]:
# 1. Garantir a ordenação por bairro e cota
df = df.sort_values(['bairro', 'nivel_m']).reset_index(drop=True)

# 2. Função de Detecção e Classificação com Tratamento de Outliers Iniciais
def classificar_anomalias_bairro(df_bairro, limiar_pct=0.005):
    b = df_bairro['affected_buildings'].values.astype(float).copy()
    n = len(b)
    tipo_anomalia = ['OK'] * n
    is_outlier = [False] * n

    # -------------------------------------------------------------
    # PASSO 0: Identificar Outliers Iniciais (Bloco de erro no início)
    # -------------------------------------------------------------
    if n >= 15:
        mediana_corpo = np.median(b[5:20]) # Mediana das cotas intermediárias
        for i in range(min(5, n)):
            if b[i] > (mediana_corpo * 5) and b[i] > 100 and mediana_corpo < 100:
                is_outlier[i] = True
                tipo_anomalia[i] = 'OUTLIER_VIA_MEDIANA'

    # -------------------------------------------------------------
    # PASSO A: Identificar Outliers Pontuais no meio da série (Spikes)
    # -------------------------------------------------------------
    for i in range(n):
        if is_outlier[i]:
            continue
        prev_val = b[i-1] if i > 0 else b[i]
        next_val = b[i+1] if i < n-1 else b[i]
        
        if b[i] > (prev_val + 50) and b[i] > (next_val + 50) and prev_val < 500:
            is_outlier[i] = True
            tipo_anomalia[i] = 'OUTLIER_VIA_MEDIANA'

    # -------------------------------------------------------------
    # PASSO B: Avaliar Quedas/Zeros por Variação Percentual (Sem Outliers)
    # -------------------------------------------------------------
    pico_acumulado = 0
    for i in range(n):
        if is_outlier[i]:
            continue
            
        val = b[i]
        
        if val < (pico_acumulado - 1) and pico_acumulado > 0:
            proximo_valido = val
            for j in range(i + 1, n):
                if not is_outlier[j] and b[j] > val:
                    proximo_valido = b[j]
                    break
            
            variacao_relativa = (proximo_valido - pico_acumulado) / pico_acumulado
            
            if variacao_relativa > limiar_pct:
                tipo_anomalia[i] = 'BURACO_VIA_INTERPOLACAO'
            else:
                tipo_anomalia[i] = 'RUIDO_VIA_ESTABILIZACAO'
        else:
            pico_acumulado = max(pico_acumulado, val)
            
    return pd.Series(tipo_anomalia, index=df_bairro.index)

# Aplica a classificação no DataFrame original (que permanece bruto)
df['tipo_anomalia'] = df.groupby('bairro', group_keys=False).apply(classificar_anomalias_bairro)
df['tem_anomalia'] = df['tipo_anomalia'] != 'OK'


# VISUALIZAÇÃO DOS ERROS

In [5]:
# Configurar Pandas para exibir todas as linhas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Total de cotas anômalas no dataset: {df['tem_anomalia'].sum()}\n")

# 1. Resumo por tipo de erro e bairro
resumo_tipos = (
    df[df['tem_anomalia']]
    .groupby(['bairro', 'tipo_anomalia'])
    .size()
    .reset_index(name='cotas_afetadas')
    .sort_values('cotas_afetadas', ascending=False)
)
print("--- 1. Diagnóstico Estatístico das Anomalias por Bairro ---")
display(resumo_tipos)

# 2. Exibição Completa das Linhas Afetadas (Usando os campos nativos do df)
tabela_auditoria = df[df['tem_anomalia']][
    ['bairro', 'nivel_m', 'affected_buildings', 'affected_population', 'tipo_anomalia']
].rename(columns={
    'affected_buildings': 'Predios_Brutos',
    'affected_population': 'Pop_Bruta'
})

print("\n--- 2. Tabela de Auditoria Bruta (Antes da Correção) ---")
display(tabela_auditoria)

Total de cotas anômalas no dataset: 75

--- 1. Diagnóstico Estatístico das Anomalias por Bairro ---


,bairro,tipo_anomalia,cotas_afetadas
0,Arquipélago,RUIDO_VIA_ESTABILIZACAO,65
2,Espírito Santo,OUTLIER_VIA_MEDIANA,6
1,Centro Histórico,BURACO_VIA_INTERPOLACAO,2
3,Lageado,BURACO_VIA_INTERPOLACAO,1
4,Rubem Berta,BURACO_VIA_INTERPOLACAO,1



--- 2. Tabela de Auditoria Bruta (Antes da Correção) ---


,bairro,nivel_m,Predios_Brutos,Pop_Bruta,tipo_anomalia
214,Arquipélago,3.1,4680,6411,RUIDO_VIA_ESTABILIZACAO
215,Arquipélago,3.2,4680,6411,RUIDO_VIA_ESTABILIZACAO
216,Arquipélago,3.3,4680,6411,RUIDO_VIA_ESTABILIZACAO
217,Arquipélago,3.4,4680,6411,RUIDO_VIA_ESTABILIZACAO
219,Arquipélago,3.6,4680,6411,RUIDO_VIA_ESTABILIZACAO
220,Arquipélago,3.7,4680,6411,RUIDO_VIA_ESTABILIZACAO
221,Arquipélago,3.8,4680,6411,RUIDO_VIA_ESTABILIZACAO
222,Arquipélago,3.9,4680,6411,RUIDO_VIA_ESTABILIZACAO
224,Arquipélago,4.1,4674,6411,RUIDO_VIA_ESTABILIZACAO
225,Arquipélago,4.2,4680,6411,RUIDO_VIA_ESTABILIZACAO


# CORREÇÃO DOS ERROS 

In [6]:

# 1. Criar a cópia limpa a partir do df bruto (sem alterar o df original)
df_limpo = df.copy()

def corrigir_bairro_adaptativo(df_bairro):
    b = df_bairro['affected_buildings'].astype(float).copy()
    tipos = df_bairro['tipo_anomalia'].values
    n = len(b)
    
    # PASSO 1: Marcar OUTLIERS e BURACOS como NaN
    for i in range(n):
        if tipos[i] in ['OUTLIER_VIA_MEDIANA', 'BURACO_VIA_INTERPOLACAO']:
            b.iloc[i] = np.nan
            
    # PASSO 2: Interpolação Linear e preenchimento de bordas
    b = b.interpolate(method='linear').bfill().ffill()
    
    # PASSO 3: Retenção da Monotonicidade em Platôs (np.maximum.accumulate)
    b = np.maximum.accumulate(b.values)
    
    return pd.Series(np.round(b).astype(int), index=df_bairro.index)

# 2. Corrigir os prédios afetados no df_limpo
df_limpo['affected_buildings'] = df_limpo.groupby('bairro', group_keys=False).apply(corrigir_bairro_adaptativo)

# 3. CÁLCULO DA FRAÇÃO DECIMAL CONTINUA (ex: 0.00021295..., 0.001, 0.015)
fracao_decimal = df_limpo['affected_buildings'] / df_limpo['total_buildings']

# 4. RECALCULO DA POPULAÇÃO AFETADA (Baseado na fração decimal contínua exata)
df_limpo['affected_population'] = np.round(df_limpo['population'] * fracao_decimal).astype(int)

# 5. RECALCULO DA PORCENTAGEM NUMÉRICA (%) (Mantém float numérico contínuo para cálculos/gráficos)
df_limpo['percentage'] = fracao_decimal * 100

print(" 'df_limpo' atualizado com sucesso: prédios, população e porcentagem recalculados numericamente!")

 'df_limpo' atualizado com sucesso: prédios, população e porcentagem recalculados numericamente!


# COMPARAÇÃO 

In [7]:
# Cruzando os dados brutos de 'df' com os dados corrigidos de 'df_limpo'
comparativo_final = pd.DataFrame({
    'bairro': df.loc[df['tem_anomalia'], 'bairro'],
    'nivel_m': df.loc[df['tem_anomalia'], 'nivel_m'],
    'Estratégia_Aplicada': df.loc[df['tem_anomalia'], 'tipo_anomalia'],
    'Predios_afetados (JSON)': df.loc[df['tem_anomalia'], 'affected_buildings'],
    'Predios_afetados_tratado (df_limpo)': df_limpo.loc[df['tem_anomalia'], 'affected_buildings']
})

print(f"Total de registros: {len(comparativo_final)}\n")
print("--- Comparativo de Resultados (df original vs. df_limpo) ---")
display(comparativo_final)

Total de registros: 75

--- Comparativo de Resultados (df original vs. df_limpo) ---


,bairro,nivel_m,Estratégia_Aplicada,Predios_afetados (JSON),Predios_afetados_tratado (df_limpo)
214,Arquipélago,3.1,RUIDO_VIA_ESTABILIZACAO,4680,4685
215,Arquipélago,3.2,RUIDO_VIA_ESTABILIZACAO,4680,4685
216,Arquipélago,3.3,RUIDO_VIA_ESTABILIZACAO,4680,4685
217,Arquipélago,3.4,RUIDO_VIA_ESTABILIZACAO,4680,4685
219,Arquipélago,3.6,RUIDO_VIA_ESTABILIZACAO,4680,4685
220,Arquipélago,3.7,RUIDO_VIA_ESTABILIZACAO,4680,4685
221,Arquipélago,3.8,RUIDO_VIA_ESTABILIZACAO,4680,4685
222,Arquipélago,3.9,RUIDO_VIA_ESTABILIZACAO,4680,4685
224,Arquipélago,4.1,RUIDO_VIA_ESTABILIZACAO,4674,4685
225,Arquipélago,4.2,RUIDO_VIA_ESTABILIZACAO,4680,4685


In [ ]:
# 1. Selecionar as colunas principais do df_limpo para inspeção visual
colunas_exibicao = [
    'bairro', 
    'nivel_m', 
    'affected_buildings', 
    'total_buildings', 
    'percentage', 
    'population', 
    'affected_population'
]

# 2. Exibir os primeiros 20 registros do df_limpo
print("--- Inspeção Visual do 'df_limpo' (Primeiras 20 linhas) ---")
display(df_limpo[colunas_exibicao].head(20))
display(df_limpo.head(20))

--- Inspeção Visual do 'df_limpo' (Primeiras 20 linhas) ---


,bairro,nivel_m,affected_buildings,total_buildings,percentage,population,affected_population
0,Aberta dos Morros,3.0,0,4696,0.000000,9854,0
1,Aberta dos Morros,3.1,0,4696,0.000000,9854,0
2,Aberta dos Morros,3.2,0,4696,0.000000,9854,0
3,Aberta dos Morros,3.3,0,4696,0.000000,9854,0
4,Aberta dos Morros,3.4,1,4696,0.021295,9854,2
5,Aberta dos Morros,3.5,1,4696,0.021295,9854,2
6,Aberta dos Morros,3.6,4,4696,0.085179,9854,8
7,Aberta dos Morros,3.7,20,4696,0.425894,9854,42
8,Aberta dos Morros,3.8,24,4696,0.511073,9854,50
9,Aberta dos Morros,3.9,27,4696,0.574957,9854,57


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population,tipo_anomalia,tem_anomalia
0,3.0,Aberta dos Morros,0,4696,0.000000,9854,0,OK,False
1,3.1,Aberta dos Morros,0,4696,0.000000,9854,0,OK,False
2,3.2,Aberta dos Morros,0,4696,0.000000,9854,0,OK,False
3,3.3,Aberta dos Morros,0,4696,0.000000,9854,0,OK,False
4,3.4,Aberta dos Morros,1,4696,0.021295,9854,2,OK,False
5,3.5,Aberta dos Morros,1,4696,0.021295,9854,2,OK,False
6,3.6,Aberta dos Morros,4,4696,0.085179,9854,8,OK,False
7,3.7,Aberta dos Morros,20,4696,0.425894,9854,42,OK,False
8,3.8,Aberta dos Morros,24,4696,0.511073,9854,50,OK,False
9,3.9,Aberta dos Morros,27,4696,0.574957,9854,57,OK,False
